# Model Comparison: Gemini 2.0 Flash (Teacher) vs Student

This notebook compares the performance of two models on the SQL generation task using **Google Gemini API**:

- **Teacher Model**: Gemini 2.0 Flash (via Google AI API)
- **Student Model**: Llama 3.1 8B fine-tuned with QLoRA

## Requirements

- Gemini API key set in `.env` (GEMINI_API_KEY)
- Configured `config/base.yaml` with `provider: gemini` and `model: gemini-2.0-flash`
- HuggingFace token for Llama 3.1 access

## Objectives

1. Validate the distillation approach by measuring quality parity
2. Compare inference latency and throughput
3. Analyze cost-effectiveness at different query volumes
4. Provide recommendations for deployment scenarios

## Hypothesis

The student model should:
- Approach teacher quality (>75% score ratio)
- Offer significantly lower cost at scale
- Provide faster inference for local deployment

In [1]:
# Cell 1: Imports, NLTK setup, and path configuration
import json
import jsonlines
import logging
import time
from pathlib import Path
from typing import Any, Dict, List

import nltk
import torch
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

# Download NLTK data to prevent import hang
print("Downloading NLTK data...")
nltk.download('punkt')  # Remove quiet=True to see progress
nltk.download('punkt_tab')  # Remove quiet=True to see progress
print("✅ NLTK data ready")

# Project imports
import sys
sys.path.append("..")

print("Importing evaluation modules...")
from src.evaluate.benchmark import BenchmarkRunner, _build_prompt
print("  ✅ benchmark")

from src.evaluate.judge import LLMJudge, JudgeExample
print("  ✅ judge")

from src.evaluate.metrics import evaluate_batch
print("  ✅ metrics")

from src.llm.client import TeacherClient, Message
print("  ✅ llm.client")

# Configuration
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Initialize model loading flag
model_loaded = False

def find_project_root() -> Path:
    """Find project root by searching for marker files."""
    MARKER_FILES = ["CLAUDE.md", "pyproject.toml", ".git"]
    current_path = Path.cwd()
    
    for parent in [current_path] + list(current_path.parents):
        for marker in MARKER_FILES:
            marker_path = parent / marker
            if marker_path.exists():
                logger.info(f"Found project root at: {parent} (marker: {marker})")
                return parent
    
    if "notebooks" in current_path.parts:
        notebooks_index = current_path.parts.index("notebooks")
        potential_root = Path(*current_path.parts[:notebooks_index])
        if (potential_root / "CLAUDE.md").exists():
            logger.info(f"Found project root via notebooks/ path: {potential_root}")
            return potential_root
    
    raise RuntimeError(f"Cannot find project root from {current_path}")

# Get project root and setup absolute paths
print("Finding project root...")
PROJECT_ROOT = find_project_root()
print(f"  ✅ Project root: {PROJECT_ROOT}")

# Constants - use absolute paths
BASE_MODEL_ID = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"
LORA_PATH = PROJECT_ROOT / "models" / "sql-llama-8b-lora"
TEST_DATA_PATH = PROJECT_ROOT / "data" / "curated" / "test.jsonl"
CONFIG_PATH = PROJECT_ROOT / "config" / "base.yaml"

# Cost constants (USD per 1M tokens) - Gemini pricing
GEMINI_INPUT_COST = 0.10
GEMINI_OUTPUT_COST = 0.40
TRAINING_COST = 0.50  # One-time training cost (local compute)

# Verify paths exist
print("Verifying paths...")
if not TEST_DATA_PATH.exists():
    raise FileNotFoundError(f"Test data not found at: {TEST_DATA_PATH}")
if not LORA_PATH.exists():
    raise FileNotFoundError(f"LoRA adapter not found at: {LORA_PATH}")
print(f"  ✅ Test data: {TEST_DATA_PATH}")
print(f"  ✅ LoRA adapter: {LORA_PATH}")
print(f"  ✅ Config: {CONFIG_PATH}")

/mnt/d/GitHub/model-tailor/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ NLTK data ready
Importing evaluation modules...


[nltk_data] Downloading package punkt to /home/jamestjy/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/jamestjy/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
2026-04-20 18:40:51,636 - INFO - Found project root at: /mnt/d/GitHub/model-tailor (marker: CLAUDE.md)


  ✅ benchmark
  ✅ judge
  ✅ metrics
  ✅ llm.client
Finding project root...
  ✅ Project root: /mnt/d/GitHub/model-tailor
Verifying paths...
  ✅ Test data: /mnt/d/GitHub/model-tailor/data/curated/test.jsonl
  ✅ LoRA adapter: /mnt/d/GitHub/model-tailor/models/sql-llama-8b-lora
  ✅ Config: /mnt/d/GitHub/model-tailor/config/base.yaml


## 1. Load Test Data

In [2]:
# Load test dataset
test_data = []
with jsonlines.open(TEST_DATA_PATH) as f:
    test_data = list(f)

print(f"Loaded {len(test_data)} test examples")
print(f"\nSample example:")
print(f"  NL:  {test_data[0]['natural_language']}")
print(f"  SQL: {test_data[0]['sql']}")
print(f"  Difficulty: {test_data[0]['difficulty']}")
print(f"  Category: {test_data[0]['category']}")

# Analyze distribution
df = pd.DataFrame(test_data)
print(f"\nDataset distribution:")
print(f"  By difficulty: {df['difficulty'].value_counts().to_dict()}")
print(f"  By category: {df['category'].value_counts().to_dict()}")

Loaded 14 test examples

Sample example:
  NL:  What is the average salary of all employees?
  SQL: SELECT AVG(salary) AS average_salary FROM employees;
  Difficulty: easy
  Category: aggregation

Dataset distribution:
  By difficulty: {'medium': 6, 'easy': 4, 'hard': 3, 'expert': 1}
  By category: {'select': 5, 'aggregation': 2, 'subquery with aggregation': 1, 'Subquery + NOT IN': 1, 'DML': 1, 'update': 1, 'CTE + Window Function (LAG)': 1, 'DML with CTE and LIMIT/OFFSET': 1, 'aggregate': 1}


## 2. Load Models

In [3]:
# Load student model (Llama 3.1 8B + LoRA)
# WARNING: This requires ~16-32GB RAM and may take several minutes
print(f"Loading base model: {BASE_MODEL_ID}")

# Clear GPU memory before loading
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("Cleared GPU cache")

import gc
gc.collect()

# Try loading with memory optimizations
try:
    print("Attempting to load model with memory optimizations...")
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
    tokenizer.pad_token = tokenizer.eos_token
    
    # Load on CPU with memory constraints
    print("⚠️  Loading model on CPU - this may take 2-5 minutes and requires 16-32GB RAM...")
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_ID,
        torch_dtype=torch.float16,  # Use float16 to save memory
        device_map="cpu",
        low_cpu_mem_usage=True,
        max_memory={0: "1GB", "cpu": "30GB"},
    )
    
    print(f"Loading LoRA adapter: {LORA_PATH}")
    student_model = PeftModel.from_pretrained(base_model, LORA_PATH)
    student_model.eval()
    
    # Get model info
    total_params = student_model.num_parameters() / 1e9
    trainable_params = sum(p.numel() for p in student_model.parameters() if p.requires_grad) / 1e6
    
    print(f"\nStudent model stats:")
    print(f"  Total parameters: {total_params:.2f}B")
    print(f"  Trainable parameters: {trainable_params:.1f}M ({trainable_params/total_params*1000:.2f}%)")
    print(f"  Device: {next(student_model.parameters()).device}")
    print(f"\n⚠️  Running on CPU - inference will be slower but functional")
    model_loaded = True
    
except Exception as e:
    print(f"\n❌ Error loading student model: {e}")
    print(f"\nFalling back - skipping student evaluation...")
    model_loaded = False
    student_model = None
    tokenizer = None

# Initialize teacher client with absolute config path
teacher_client = TeacherClient(config_path=str(CONFIG_PATH))
print(f"\nTeacher client initialized: {teacher_client.provider} / {teacher_client.model}")

if not model_loaded:
    print("\n⚠️  Student model could not be loaded. Only teacher evaluation will be performed.")
    print("Possible solutions:")
    print("  1. Free up system RAM (close other applications)")
    print("  2. Use a machine with more RAM (recommended: 32GB+)")
    print("  3. Use GGUF quantized model with llama.cpp")
    print("  4. Run on cloud GPU with sufficient VRAM")

Loading base model: unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit
Cleared GPU cache
Attempting to load model with memory optimizations...


2026-04-20 18:41:28,090 - INFO - HTTP Request: HEAD https://huggingface.co/unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-04-20 18:41:28,098 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit/f15c379fb32bb402fa06a7ae9aecb1febf4b79ec/config.json "HTTP/1.1 200 OK"
2026-04-20 18:41:28,418 - INFO - HTTP Request: HEAD https://huggingface.co/unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-04-20 18:41:28,426 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit/f15c379fb32bb402fa06a7ae9aecb1febf4b79ec/tokenizer_config.json "HTTP/1.1 200 OK"
2026-04-20 18:41:28,683 - INFO - HTTP Request: GET https://huggingface.co/api/models/unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit/tree/main/additional_chat_templates?recursive=false&expand=false 

⚠️  Loading model on CPU - this may take 2-5 minutes and requires 16-32GB RAM...


2026-04-20 18:41:30,838 - INFO - HTTP Request: HEAD https://huggingface.co/unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-04-20 18:41:30,847 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit/f15c379fb32bb402fa06a7ae9aecb1febf4b79ec/config.json "HTTP/1.1 200 OK"
`torch_dtype` is deprecated! Use `dtype` instead!
2026-04-20 18:41:31,154 - INFO - HTTP Request: HEAD https://huggingface.co/unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-04-20 18:41:31,162 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit/f15c379fb32bb402fa06a7ae9aecb1febf4b79ec/config.json "HTTP/1.1 200 OK"
Loading weights: 100%|██████████| 291/291 [00:05<00:00, 49.49it/s]
2026-04-20 18:41:49,333 - INFO - HTTP Request: HEAD https://huggingface.co/unsloth/Meta-Llama-3.1

Loading LoRA adapter: /mnt/d/GitHub/model-tailor/models/sql-llama-8b-lora

Student model stats:
  Total parameters: 8.07B
  Trainable parameters: 0.0M (0.00%)
  Device: cpu

⚠️  Running on CPU - inference will be slower but functional

Teacher client initialized: gemini / gemini-2.0-flash


## 3. Evaluate Teacher Model (Gemini 2.0 Flash)

In [4]:
def evaluate_teacher(test_data: List[Dict], client: TeacherClient) -> tuple:
    """Evaluate Sonnet on test set.
    
    Returns:
        (predictions, elapsed_time, cost_breakdown)
    """
    predictions = []
    total_input_tokens = 0
    total_output_tokens = 0
    
    t0 = time.perf_counter()
    
    print(f"Evaluating teacher model on {len(test_data)} examples...")
    
    for i, example in enumerate(test_data):
        prompt = _build_prompt(example["natural_language"])
        messages = [Message(role="user", content=prompt)]
        
        try:
            response = client.complete(messages, temperature=0.0, max_tokens=256)
            
            # Extract SQL (first line, remove trailing semicolon)
            sql = response.strip().split("\n")[0].strip().rstrip(";")
            predictions.append(sql)
            
            # Estimate tokens (rough approximation: 1 token ≈ 4 chars)
            total_input_tokens += len(prompt) // 4
            total_output_tokens += len(response) // 4
            
            if (i + 1) % 5 == 0:
                print(f"  Progress: {i + 1}/{len(test_data)} examples")
                
        except Exception as e:
            logger.error(f"Error on example {i}: {e}")
            predictions.append("")
    
    elapsed = time.perf_counter() - t0
    
    # Calculate API cost
    input_cost = (total_input_tokens / 1_000_000) * GEMINI_INPUT_COST
    output_cost = (total_output_tokens / 1_000_000) * GEMINI_OUTPUT_COST
    total_cost = input_cost + output_cost
    
    cost_breakdown = {
        "input_tokens": total_input_tokens,
        "output_tokens": total_output_tokens,
        "input_cost_usd": round(input_cost, 4),
        "output_cost_usd": round(output_cost, 4),
        "total_cost_usd": round(total_cost, 4),
    }
    
    print(f"\nTeacher evaluation complete: {elapsed:.1f}s, ${total_cost:.4f}")
    return predictions, elapsed, cost_breakdown

# Run teacher evaluation
teacher_predictions, teacher_time, teacher_cost = evaluate_teacher(test_data, teacher_client)

print(f"\nTeacher results:")
print(f"  Time: {teacher_time:.2f}s")
print(f"  Throughput: {len(test_data)/teacher_time:.3f} examples/sec")
print(f"  Cost: ${teacher_cost['total_cost_usd']:.4f}")
print(f"  Avg latency: {teacher_time/len(test_data):.3f}s per query")

2026-04-20 18:43:09,804 - INFO - AFC is enabled with max remote calls: 10.
2026-04-20 18:43:09,912 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent "HTTP/1.1 429 Too Many Requests"
2026-04-20 18:43:09,914 - ERROR - Error on example 0: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model:

Evaluating teacher model on 14 examples...


2026-04-20 18:43:10,047 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent "HTTP/1.1 429 Too Many Requests"
2026-04-20 18:43:10,049 - ERROR - Error on example 2: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\nPlease retry in 50.026836668s.', 'status': 'RESOURCE_EXH


Teacher evaluation complete: 1.0s, $0.0000

Teacher results:
  Time: 1.00s
  Throughput: 14.053 examples/sec
  Cost: $0.0000
  Avg latency: 0.071s per query


In [ ]:
# Check first few predictions
print("First 5 predictions:")
for i in range(5):
    print(f"{i+1}. {teacher_predictions[i]}")

First 5 predictions:
1. 
2. 
3. 
4. 
5. 


: 

## 4. Evaluate Student Model (Llama)

In [ ]:
def evaluate_student(test_data: List[Dict], model: Any, tokenizer: Any) -> tuple:
    """Evaluate Llama student model on test set.
    
    Returns:
        (predictions, elapsed_time)
    """
    prompts = [_build_prompt(ex["natural_language"]) for ex in test_data]
    predictions = []
    
    t0 = time.perf_counter()
    
    print(f"Evaluating student model on {len(test_data)} examples...")
    
    batch_size = 2  # Very conservative batch size for CPU
    
    for start in range(0, len(prompts), batch_size):
        batch_prompts = prompts[start:start + batch_size]
        inputs = tokenizer(
            batch_prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512,  # Reduced to save memory
        ).to(model.device)
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=128,  # Reduced for SQL queries
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
            )
        
        # Decode and strip prompt
        prompt_lengths = inputs["input_ids"].shape[1]
        for output in outputs:
            generated = output[prompt_lengths:]
            decoded = tokenizer.decode(generated, skip_special_tokens=True)
            sql = decoded.strip().split("\n")[0].strip().rstrip(";")
            predictions.append(sql)
        
        # Clear cache after each batch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        
        if (start + batch_size) % 2 == 0 or start + batch_size >= len(prompts):
            print(f"  Progress: {min(start + batch_size, len(prompts))}/{len(prompts)} examples")
    
    elapsed = time.perf_counter() - t0
    print(f"\nStudent evaluation complete: {elapsed:.1f}s")
    return predictions, elapsed

# Check if student model was loaded successfully
if not model_loaded:
    print("\n⚠️  Skipping student evaluation - model not loaded.")
    print("Only teacher model results will be available.")
    student_predictions = ["" for _ in test_data]
    student_time = 0
else:
    # Run student evaluation
    student_predictions, student_time = evaluate_student(test_data, student_model, tokenizer)

    print(f"\nStudent results:")
    print(f"  Time: {student_time:.2f}s")
    print(f"  Throughput: {len(test_data)/student_time:.3f} examples/sec")
    print(f"  Cost: $0.0000 (local inference)")
    print(f"  Avg latency: {student_time/len(test_data):.3f}s per query")

## 5. Compare Prediction Quality

In [ ]:
# Compute traditional metrics
targets = [ex["sql"] for ex in test_data]

teacher_metrics = evaluate_batch(
    teacher_predictions, targets,
    db_path=None,
    metrics=["exact_match", "bleu", "rouge"]
)

student_metrics = evaluate_batch(
    student_predictions, targets,
    db_path=None,
    metrics=["exact_match", "bleu", "rouge"]
)

# Create comparison dataframe
comparison_data = []
for metric in teacher_metrics.keys():
    teacher_score = teacher_metrics[metric]
    student_score = student_metrics[metric]
    delta = student_score - teacher_score
    pct_of_teacher = (student_score / teacher_score * 100) if teacher_score > 0 else 0
    
    comparison_data.append({
        "Metric": metric.upper(),
        "Teacher (Gemini)": f"{teacher_score:.4f}",
        "Student (Llama)": f"{student_score:.4f}",
        "Delta": f"{delta:+.4f}",
        "% of Teacher": f"{pct_of_teacher:.1f}%"
    })

comparison_df = pd.DataFrame(comparison_data)
print("\n=== Quality Metrics Comparison ===")
print(comparison_df.to_string(index=False))

# Calculate overall quality parity
avg_parity = sum(
    student_metrics[m] / teacher_metrics[m] * 100
    for m in teacher_metrics.keys()
    if teacher_metrics[m] > 0
) / len([m for m in teacher_metrics.keys() if teacher_metrics[m] > 0])

print(f"\nOverall Quality Parity: {avg_parity:.1f}%")

if avg_parity >= 90:
    verdict = "EXCELLENT - Student nearly matches teacher"
elif avg_parity >= 75:
    verdict = "GOOD - Student captures most teacher capability"
elif avg_parity >= 60:
    verdict = "ACCEPTABLE - Student shows reasonable distillation"
else:
    verdict = "NEEDS WORK - Significant quality gap"
    
print(f"Verdict: {verdict}")

## 6. LLM Judge Evaluation

In [ ]:
# Initialize judge (uses GLM 4.7 from config)
judge = LLMJudge()

print("Running LLM judge evaluation...")
print("Note: This will make API calls to score each prediction.")

# Prepare judge examples
teacher_examples = [
    JudgeExample(nl=ex["natural_language"], pred_sql=pred, target_sql=ex["sql"])
    for ex, pred in zip(test_data, teacher_predictions)
]

student_examples = [
    JudgeExample(nl=ex["natural_language"], pred_sql=pred, target_sql=ex["sql"])
    for ex, pred in zip(test_data, student_predictions)
]

# Judge both sets (concurrency=2 for API rate limits)
teacher_judge_results = judge.judge_batch(teacher_examples, concurrency=2)
student_judge_results = judge.judge_batch(student_examples, concurrency=2)

# Calculate mean scores
teacher_mean = judge.mean_score(teacher_judge_results)
student_mean = judge.mean_score(student_judge_results)

print(f"\n=== LLM Judge Scores (GLM 4.7) ===")
print(f"Teacher (Gemini):  {teacher_mean:.3f}/5.0")
print(f"Student (Llama):   {student_mean:.3f}/5.0")
print(f"Delta:             {student_mean - teacher_mean:+.3f}")
print(f"Quality Ratio:     {(student_mean/teacher_mean*100):.1f}%")

## 7. Performance Comparison

In [ ]:
# Performance metrics
perf_data = {
    "Metric": [
        "Total Time (s)",
        "Throughput (examples/s)",
        "Avg Latency (s/query)",
        "Cost (USD)"
    ],
    "Teacher (Gemini)": [
        f"{teacher_time:.2f}",
        f"{len(test_data)/teacher_time:.3f}",
        f"{teacher_time/len(test_data):.3f}",
        f"${teacher_cost['total_cost_usd']:.4f}"
    ],
    "Student (Llama)": [
        f"{student_time:.2f}",
        f"{len(test_data)/student_time:.3f}",
        f"{student_time/len(test_data):.3f}",
        "$0.0000"
    ]
}

perf_df = pd.DataFrame(perf_data)
print("\n=== Performance Comparison ===")
print(perf_df.to_string(index=False))

# Speedup calculation
speedup = teacher_time / student_time if student_time > 0 else 0
print(f"\nSpeedup: {speedup:.2f}x {'(teacher faster)' if speedup < 1 else '(student faster)'}")

## 8. Cost Analysis

In [ ]:
# Cost calculations
teacher_per_query = teacher_cost['total_cost_usd'] / len(test_data)
break_even_queries = int(TRAINING_COST / teacher_per_query) if teacher_per_query > 0 else float('inf')

# Project costs for different volumes
volumes = [10, 100, 1000, 10000]
cost_projections = []

for volume in volumes:
    teacher_total = teacher_per_query * volume
    student_total = TRAINING_COST  # One-time cost
    savings = teacher_total - student_total
    
    cost_projections.append({
        "Queries": volume,
        "Teacher Cost": f"${teacher_total:.2f}",
        "Student Cost": f"${student_total:.2f}",
        "Savings": f"${savings:.2f}",
        "ROI": f"{savings/TRAINING_COST*100:.0f}%"
    })

cost_df = pd.DataFrame(cost_projections)

print("\n=== Cost Analysis ===")
print(f"\nPer-query costs:")
print(f"  Teacher (Gemini):  ${teacher_per_query:.4f} per query")
print(f"  Student (Llama):   $0.0000 (local, after one-time training cost)")
print(f"  One-time training: ${TRAINING_COST:.2f}")
print(f"\nBreak-even point: {break_even_queries} queries")

print(f"\nProjected costs by volume:")
print(cost_df.to_string(index=False))

print(f"\nKey insight: At {break_even_queries} queries, the student model becomes more cost-effective.")
print(f"For high-volume applications (1000+ queries), savings are significant.")

## 9. Visualization

In [ ]:
# Create visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Model Comparison: Sonnet vs Llama 3.1 8B', fontsize=16, fontweight='bold')

# 1. Quality Metrics Bar Chart
ax1 = axes[0, 0]
metrics_list = list(teacher_metrics.keys())
teacher_scores = [teacher_metrics[m] for m in metrics_list]
student_scores = [student_metrics[m] for m in metrics_list]

x = range(len(metrics_list))
width = 0.35

ax1.bar([i - width/2 for i in x], teacher_scores, width, label='Teacher (Gemini)', alpha=0.8)
ax1.bar([i + width/2 for i in x], student_scores, width, label='Student (Llama)', alpha=0.8)
ax1.set_xlabel('Metric')
ax1.set_ylabel('Score')
ax1.set_title('Quality Metrics Comparison')
ax1.set_xticks(x)
ax1.set_xticklabels([m.upper() for m in metrics_list])
ax1.legend()
ax1.grid(axis='y', alpha=0.3)

# 2. LLM Judge Scores
ax2 = axes[0, 1]
judge_scores = [teacher_mean, student_mean]
labels = ['Teacher\n(Sonnet)', 'Student\n(Llama)']
colors = ['#2E86AB', '#A23B72']

bars = ax2.bar(labels, judge_scores, color=colors, alpha=0.8)
ax2.set_ylabel('Score (1-5)')
ax2.set_title('LLM Judge Evaluation (GLM 4.7)')
ax2.set_ylim(0, 5)
ax2.grid(axis='y', alpha=0.3)

# Add score labels on bars
for bar, score in zip(bars, judge_scores):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
             f'{score:.2f}', ha='center', va='bottom', fontweight='bold')

# 3. Performance Comparison (Log Scale)
ax3 = axes[1, 0]
perf_metrics = ['Avg Latency\n(s/query)', 'Cost per Query\n(USD)']
teacher_perf = [teacher_time/len(test_data), teacher_per_query]
student_perf = [student_time/len(test_data), 0.0001]  # Small value for log scale

x = range(len(perf_metrics))
ax3.bar([i - width/2 for i in x], teacher_perf, width, label='Teacher (Gemini)', alpha=0.8)
ax3.bar([i + width/2 for i in x], student_perf, width, label='Student (Llama)', alpha=0.8)
ax3.set_xlabel('Performance Metric')
ax3.set_ylabel('Value (log scale)')
ax3.set_title('Performance Comparison')
ax3.set_xticks(x)
ax3.set_xticklabels(perf_metrics)
ax3.set_yscale('log')
ax3.legend()
ax3.grid(axis='y', alpha=0.3)

# 4. Cost Projection by Volume
ax4 = axes[1, 1]
volumes_plot = volumes
teacher_costs = [teacher_per_query * v for v in volumes]
student_costs = [TRAINING_COST] * len(volumes)

ax4.plot(volumes_plot, teacher_costs, 'o-', label='Teacher (Gemini)', linewidth=2, markersize=8)
ax4.plot(volumes_plot, student_costs, 's-', label='Student (Llama)', linewidth=2, markersize=8)
ax4.axvline(x=break_even_queries, color='red', linestyle='--', alpha=0.5, label=f'Break-even: {break_even_queries} queries')
ax4.set_xlabel('Number of Queries')
ax4.set_ylabel('Total Cost (USD)')
ax4.set_title('Cost Projection by Query Volume')
ax4.set_xscale('log')
ax4.set_yscale('log')
ax4.legend()
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('tasks/sql_generation/comparison_visualization_gemini.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nVisualization saved to: tasks/sql_generation/comparison_visualization_gemini.png")

## 10. Summary and Recommendations

In [ ]:
# Compile summary statistics
summary = {
    "quality_parity_percent": round(avg_parity, 1),
    "teacher_judge_score": round(teacher_mean, 3),
    "student_judge_score": round(student_mean, 3),
    "speedup": round(student_time / teacher_time, 2) if teacher_time > 0 else 0,
    "teacher_latency_ms": round(teacher_time / len(test_data) * 1000, 1),
    "student_latency_ms": round(student_time / len(test_data) * 1000, 1),
    "teacher_cost_per_query": round(teacher_per_query, 4),
    "break_even_queries": break_even_queries,
    "training_cost_usd": TRAINING_COST,
}

print("\n" + "="*70)
print("COMPARISON SUMMARY - Sonnet vs Llama 3.1 8B")
print("="*70)

print(f"\n📊 Quality:")
print(f"  Student achieves {summary['quality_parity_percent']}% of teacher quality")
print(f"  Judge scores: Teacher={summary['teacher_judge_score']:.2f}, Student={summary['student_judge_score']:.2f}")

print(f"\n⚡ Performance:")
print(f"  Speedup: {summary['speedup']:.2f}x")
print(f"  Latency: Teacher={summary['teacher_latency_ms']:.1f}ms, Student={summary['student_latency_ms']:.1f}ms")

print(f"\n💰 Cost:")
print(f"  Teacher (Gemini): ${summary['teacher_cost_per_query']:.4f} per query")
print(f"  Student (Llama):    ${summary['training_cost_usd']:.2f} one-time + $0.00 per query")
print(f"  Break-even: {summary['break_even_queries']} queries")

print(f"\n" + "="*70)
print("RECOMMENDATIONS")
print("="*70)

print(f"\n✅ Use the Student Model (Llama 3.1 8B) if:")
print(f"  - You expect {summary['break_even_queries']}+ queries (cost-effective)")
print(f"  - You need low-latency local inference")
print(f"  - Data privacy requires on-premise deployment")
print(f"  - You can accept {100 - summary['quality_parity_percent']:.1f}% quality difference")

print(f"\n✅ Use the Teacher Model (Sonnet) if:")
print(f"  - You need the highest possible quality")
print(f"  - Query volume is low (< {summary['break_even_queries']} queries)")
print(f"  - You don't want to manage model infrastructure")
print(f"  - API-based deployment is preferred")

print(f"\n🎯 Key Finding:")
if summary['quality_parity_percent'] >= 75:
    print(f"  The distillation approach is SUCCESSFUL. The student model captures")
    print(f"  most of the teacher's capability at a fraction of the cost for scale.")
else:
    print(f"  The distillation shows promise but needs improvement. Consider:")
    print(f"  - More training data (currently {len(test_data)} test examples)")
    print(f"  - Additional training epochs")
    print(f"  - Hyperparameter tuning")

print("\n" + "="*70)

## 11. Save Results

In [ ]:
# Save detailed results to JSON
results = {
    "metadata": {
        "test_examples": len(test_data),
        "teacher_model": "gemini-2.0-flash",
        "student_model": "Meta-Llama-3.1-8B + LoRA",
        "judge_model": "glm-4.7",  # Judge still uses GLM from config
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S")
    },
    "teacher": {
        "predictions": teacher_predictions,
        "elapsed_seconds": round(teacher_time, 2),
        "cost_breakdown": teacher_cost,
        "metrics": teacher_metrics
    },
    "student": {
        "predictions": student_predictions,
        "elapsed_seconds": round(student_time, 2),
        "training_cost_usd": TRAINING_COST,
        "metrics": student_metrics
    },
    "llm_judge": {
        "teacher_score": round(teacher_mean, 3),
        "student_score": round(student_mean, 3),
        "quality_parity_percent": round(avg_parity, 1)
    },
    "summary": summary
}

# Save results
output_path = Path("tasks/sql_generation/comparison_results_gemini.json")
output_path.parent.mkdir(parents=True, exist_ok=True)

with open(output_path, "w") as f:
    json.dump(results, f, indent=2)

print(f"\nDetailed results saved to: {output_path}")
print(f"Visualization saved to: tasks/sql_generation/comparison_visualization_gemini.png")